In [1]:
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    host="db",          # nombre del servicio, no localhost
    port=5432,          # puerto interno del contenedor
    database=os.environ["POSTGRES_DB"],
)
engine_pg = create_engine(url)

with engine_pg.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [2]:
import pandas as pd

RUTA_CSV = "/home/jovyan/work/data/sensores.csv"

df = pd.read_csv(RUTA_CSV, parse_dates=["fecha_hora"])

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df.head()

(192, 5)
edificio                  str
fecha_hora     datetime64[us]
temperatura           float64
humedad               float64
co2_ppm                 int64
dtype: object
edificio       0
fecha_hora     0
temperatura    0
humedad        0
co2_ppm        0
dtype: int64


,edificio,fecha_hora,temperatura,humedad,co2_ppm
0,A,2026-10-14 08:00:00,21.9,43.7,762
1,B,2026-10-14 08:00:00,23.1,51.9,873
2,C,2026-10-14 08:00:00,23.5,49.7,1221
3,D,2026-10-14 08:00:00,22.6,54.0,1277
4,A,2026-10-14 08:15:00,21.8,44.5,599


In [4]:
from sqlalchemy import text

def cargar_lecturas(df, engine):
    """Carga el DataFrame en la tabla 'lecturas' (la sustituye si ya existe)."""
    df.to_sql("lecturas", engine, if_exists="replace", index=False)

def contar_lecturas(engine):
    """Devuelve cuántas filas hay en la tabla 'lecturas'."""
    with engine.connect() as conn:
        return conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

In [8]:
cargar_lecturas(df, engine_pg)
print("Registros en PostgreSQL:", contar_lecturas(engine_pg))

Registros en PostgreSQL: 192


In [6]:
pd.read_sql("SELECT * FROM lecturas LIMIT 5", engine_pg)

,edificio,fecha_hora,temperatura,humedad,co2_ppm
0,A,2026-10-14 08:00:00,21.9,43.7,762
1,B,2026-10-14 08:00:00,23.1,51.9,873
2,C,2026-10-14 08:00:00,23.5,49.7,1221
3,D,2026-10-14 08:00:00,22.6,54.0,1277
4,A,2026-10-14 08:15:00,21.8,44.5,599


In [7]:
pd.read_sql(
    "SELECT column_name, data_type FROM information_schema.columns "
    "WHERE table_name = 'lecturas'",
    engine_pg,
)

,column_name,data_type
0,fecha_hora,timestamp without time zone
1,temperatura,double precision
2,humedad,double precision
3,co2_ppm,bigint
4,edificio,text


In [9]:
engine_sqlite = create_engine("sqlite:////home/jovyan/work/data/smartcampus.db")

cargar_lecturas(df, engine_sqlite)
print("Registros en SQLite:", contar_lecturas(engine_sqlite))

Registros en SQLite: 192


In [10]:
def temperatura_media_por_edificio(engine):
    consulta = """
        SELECT edificio, AVG(temperatura) AS temperatura_media
        FROM lecturas
        GROUP BY edificio
        ORDER BY edificio
    """
    return pd.read_sql(consulta, engine).round(2)

def edificio_mayor_co2(engine):
    consulta = """
        SELECT edificio, AVG(co2_ppm) AS co2_medio
        FROM lecturas
        GROUP BY edificio
        ORDER BY co2_medio DESC
        LIMIT 1
    """
    return pd.read_sql(consulta, engine).round(2)

In [11]:
for nombre, eng in [("PostgreSQL", engine_pg), ("SQLite", engine_sqlite)]:
    print(f"===== {nombre} =====")
    print("Registros cargados:", contar_lecturas(eng))
    print("\nTemperatura media por edificio:")
    print(temperatura_media_por_edificio(eng).to_string(index=False))
    print("\nEdificio con mayor CO2 medio:")
    print(edificio_mayor_co2(eng).to_string(index=False))
    print()

===== PostgreSQL =====
Registros cargados: 192

Temperatura media por edificio:
edificio  temperatura_media
       A              22.62
       B              23.68
       C              24.44
       D              23.08

Edificio con mayor CO2 medio:
edificio  co2_medio
       D    1349.29

===== SQLite =====
Registros cargados: 192

Temperatura media por edificio:
edificio  temperatura_media
       A              22.62
       B              23.68
       C              24.44
       D              23.08

Edificio con mayor CO2 medio:
edificio  co2_medio
       D    1349.29



In [12]:
import requests

API = "http://api:5000"

# Lectura válida: debe devolver 201
valida = {
    "edificio": "A",
    "fecha_hora": "2026-10-15 12:34:56",
    "temperatura": 23.7,
    "humedad": 51.0,
    "co2_ppm": 1375,
}
r = requests.post(f"{API}/lecturas", json=valida)
print(r.status_code, r.json())

# Lectura incompleta (falta co2_ppm): debe devolver 400
incompleta = {
    "edificio": "A",
    "fecha_hora": "2026-10-15 12:36:10",
    "temperatura": 23.5,
    "humedad": 45.0,
}
r = requests.post(f"{API}/lecturas", json=incompleta)
print(r.status_code, r.json())

201 {'lectura': {'co2_ppm': 1375, 'edificio': 'A', 'fecha_hora': '2026-10-15 12:34:56', 'humedad': 51.0, 'temperatura': 23.7}, 'mensaje': 'Lectura guardada'}
400 {'campos_faltantes': ['co2_ppm'], 'error': 'Faltan campos obligatorios'}


In [13]:
print("Registros en PostgreSQL:", contar_lecturas(engine_pg))

Registros en PostgreSQL: 193
